<a href="https://colab.research.google.com/github/Nanayeb34/IndabaX-2026/blob/main/notebook/Part2.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# The Audit
### Building Ethical Vision Systems — Ghana Data Science Summit 2026
*Part 2 · Led by Emmanuel Asante*

---

A health startup in Accra has built a skin condition detection app for community health workers in rural Ghana. The model has been trained. The benchmark numbers look strong. But who was in the training data? And will it work for the patients in Brong-Ahafo?

**Your job:** run the audit, document the failures, and make a recommendation.

**How to use this notebook:**
1. Run the four setup cells below — they install dependencies, download the data, and define all helper functions
2. Work through each section top to bottom
3. Stop at every `✍️ Reflection` block and write your answers in the markdown cell below it
4. Your final deliverable is a recommendation at the end of Section 04


---
## ⚙️ SECTION 00 — SETUP
*Run these four cells once. Then you're ready.*

---

In [ ]:
# Install dependencies — takes 2–4 minutes on a fresh Colab instance
%pip install -q torch torchvision timm numpy pandas matplotlib seaborn \
             scikit-learn pillow tqdm gdown
print("✓ Dependencies ready")

In [ ]:
# Clone the repo (for the HAM10000 datasheet and config files)
import os
if not os.path.exists("IndabaX-2026"):
    !git clone https://github.com/Nanayeb34/IndabaX-2026.git IndabaX-2026 --depth 1 -q
    print("✓ Repo cloned")
else:
    print("✓ Repo already present")

# ── Download audit data from Google Drive ────────────────────────────────────
# The facilitator has prepared a Drive folder with:
#   data/ham10000_test/images/       — HAM10000 held-out test images
#   data/ham10000_test/HAM10000_metadata.csv
#   data/african_context/images/     — African-context stress-test images
#   data/african_context/african_context_labels.csv
#   model/efficientnet_b0_ham10000.pth
#
# Replace AUDIT_FOLDER_ID below with the shared folder ID from the facilitator.
# ─────────────────────────────────────────────────────────────────────────────

import gdown
from pathlib import Path

AUDIT_FOLDER_ID = "REPLACE_WITH_FOLDER_ID"   # ← facilitator fills this in

DATA_DIR  = Path("audit_data")
MODEL_DIR = Path("audit_data/model")
DATA_DIR.mkdir(exist_ok=True)

if AUDIT_FOLDER_ID != "REPLACE_WITH_FOLDER_ID":
    try:
        gdown.download_folder(
            f"https://drive.google.com/drive/folders/{AUDIT_FOLDER_ID}",
            output=str(DATA_DIR), quiet=False
        )
        print("✓ Audit data downloaded")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Ask the facilitator for the folder ID, or see the README.")
else:
    print("⚠ AUDIT_FOLDER_ID not set — replace it with the value from the facilitator.")
    print("  Without the data, inference cells will be skipped gracefully.")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, zipfile, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

# ── Palette ──────────────────────────────────────────────────────────────────
TEAL  = "#028090"
NAVY  = "#1A2535"
AMBER = "#D97706"
RED   = "#DC2626"
GREY  = "#9CA3AF"

# ── Class definitions ─────────────────────────────────────────────────────────
CLASS_NAMES  = ["nv", "mel", "bcc", "akiec", "bkl", "df", "vasc"]
CLASS_LABELS = {
    "nv":    "Melanocytic
Nevi",
    "mel":   "Melanoma",
    "bcc":   "Basal Cell
Carcinoma",
    "akiec": "Actinic
Keratoses",
    "bkl":   "Benign
Keratoses",
    "df":    "Dermatofibroma",
    "vasc":  "Vascular
Lesions",
}
CLASS_LABELS_FLAT = {k: v.replace("
", " ") for k, v in CLASS_LABELS.items()}

MALIGNANT = {"mel", "bcc", "akiec"}

# ── Transform — identical for both datasets ───────────────────────────────────
TRANSFORM = T.Compose([
    T.Resize(224),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Dataset ───────────────────────────────────────────────────────────────────
class SkinDataset(Dataset):
    # Works for both HAM10000 test split and African-context dataset
    def __init__(self, images_dir, csv_path, img_col="image_id", label_col="dx"):
        self.images_dir = Path(images_dir)
        df = pd.read_csv(csv_path)
        df["_path"] = df[img_col].apply(lambda x: self.images_dir / f"{x}.jpg")
        df = df[df["_path"].apply(lambda p: p.exists())].reset_index(drop=True)
        self.df         = df
        self.image_ids  = df[img_col].tolist()
        self.labels     = [CLASS_NAMES.index(dx) for dx in df[label_col]]
        self.fitzpatrick = df["fitzpatrick_type"].tolist() if "fitzpatrick_type" in df.columns else [None]*len(df)

    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i]["_path"]).convert("RGB")
        return TRANSFORM(img), self.labels[i], self.image_ids[i]


def make_loader(images_dir, csv_path, batch_size=32, num_workers=2):
    ds = SkinDataset(images_dir, csv_path)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=num_workers, pin_memory=True), ds


# ── Model loading ─────────────────────────────────────────────────────────────
def load_model(checkpoint_path, device="cpu"):
    # Load EfficientNet-B0 fine-tuned on HAM10000
    model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=7)
    ckpt = torch.load(checkpoint_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(state)
    model.to(device).eval()
    print(f"✓ Model loaded from {Path(checkpoint_path).name}")
    print(f"  Architecture: EfficientNet-B0  |  Classes: {len(CLASS_NAMES)}  |  Device: {device}")
    return model


# ── Inference ─────────────────────────────────────────────────────────────────
def run_inference(model, loader, device):
    # Run a full inference pass. Returns dict of predictions/labels/probs/ids.
    preds, probs_all, labels_all, ids_all = [], [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, lbls, img_ids in tqdm(loader, desc="Inference", unit="batch"):
            p = torch.softmax(model(imgs.to(device)), 1)
            preds.extend(p.argmax(1).cpu().tolist())
            probs_all.extend(p.cpu().tolist())
            labels_all.extend(lbls.tolist())
            ids_all.extend(list(img_ids))
    return {"predictions": preds, "probabilities": probs_all,
            "true_labels": labels_all, "image_ids": ids_all}


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(preds, true_labels, probs):
    p, y = np.array(preds), np.array(true_labels)
    pr   = np.array(probs)
    per_cls_acc = {}
    for i, cls in enumerate(CLASS_NAMES):
        m = y == i
        per_cls_acc[cls] = float(accuracy_score(y[m], p[m])) if m.sum() > 0 else float("nan")
    pcf = f1_score(y, p, average=None, zero_division=0, labels=list(range(7)))
    max_p = pr.max(1)
    correct = p == y
    return {
        "overall_accuracy":      float(accuracy_score(y, p)),
        "macro_f1":              float(f1_score(y, p, average="macro",    zero_division=0)),
        "weighted_f1":           float(f1_score(y, p, average="weighted", zero_division=0)),
        "per_class_accuracy":    per_cls_acc,
        "per_class_f1":          {cls: float(pcf[i]) for i, cls in enumerate(CLASS_NAMES)},
        "confusion_matrix":      confusion_matrix(y, p, labels=list(range(7))),
        "avg_confidence_correct": float(max_p[correct].mean()) if correct.any() else float("nan"),
        "avg_confidence_wrong":  float(max_p[~correct].mean()) if (~correct).any() else float("nan"),
    }


def build_failure_table(preds, true_labels, probs, image_ids, conf_thresh=0.80):
    rows = []
    for img_id, pred, true, prob in zip(image_ids, preds, true_labels, probs):
        conf     = float(max(prob))
        pred_cls = CLASS_NAMES[pred]
        true_cls = CLASS_NAMES[true]
        if pred == true:
            etype = "correct"
        elif conf >= conf_thresh:
            etype = "high_conf_wrong"
        elif true_cls in MALIGNANT and pred_cls not in MALIGNANT:
            etype = "false_negative"
        elif true_cls not in MALIGNANT and pred_cls in MALIGNANT:
            etype = "false_positive"
        else:
            etype = "wrong_other"
        rows.append({
            "image_id": img_id,
            "true_label": CLASS_LABELS_FLAT[true_cls],
            "predicted_label": CLASS_LABELS_FLAT[pred_cls],
            "confidence": round(conf, 4),
            "error_type": etype,
        })
    return pd.DataFrame(rows).sort_values("confidence", ascending=False).reset_index(drop=True)


# ── Plotting helpers ──────────────────────────────────────────────────────────
def _style():
    plt.style.use("seaborn-v0_8-whitegrid")

def plot_confusion_matrix(true_labels, preds, title="Confusion Matrix", figsize=(9,7)):
    _style()
    labels = [CLASS_LABELS[c].replace("
"," ") for c in CLASS_NAMES]
    cm = confusion_matrix(true_labels, preds, labels=list(range(7)))
    cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(1, keepdims=True))
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm_norm, annot=True, fmt=".2f",
                cmap=sns.light_palette(TEAL, as_cmap=True),
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.5, linecolor="#F3F4F6", cbar_kws={"shrink":0.8})
    ax.set_title(title, fontsize=14, fontweight="bold", color=NAVY, pad=12)
    ax.set_ylabel("True Label", fontsize=11, color=NAVY)
    ax.set_xlabel("Predicted Label", fontsize=11, color=NAVY)
    ax.tick_params(labelsize=8)
    plt.tight_layout(); plt.show()

def plot_per_class_f1(metrics, title="Per-Class F1 Score", figsize=(9,4)):
    _style()
    items   = sorted(metrics["per_class_f1"].items(), key=lambda x: x[1], reverse=True)
    classes, scores = zip(*items)
    names   = [CLASS_LABELS_FLAT[c] for c in classes]
    colours = [AMBER if s < 0.5 else TEAL for s in scores]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(names, scores, color=colours, edgecolor="white")
    for bar, s in zip(bars, scores):
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                f"{s:.2f}", va="center", ha="left", fontsize=9, color=NAVY)
    ax.set_xlim(0, 1.15); ax.set_xlabel("F1 Score", color=NAVY)
    ax.set_title(title, fontsize=13, fontweight="bold", color=NAVY, pad=10)
    ax.axvline(0.5, color=AMBER, ls="--", lw=1, alpha=0.7, label="F1 = 0.50")
    ax.legend(fontsize=8); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

def plot_metric_comparison(m_a, m_b, label_a="HAM10000", label_b="African Context", figsize=(14,6)):
    _style()
    names   = ["Overall
Accuracy", "Macro
F1"] + [CLASS_LABELS[c] for c in CLASS_NAMES]
    vals_a  = [m_a["overall_accuracy"], m_a["macro_f1"]] + [m_a["per_class_f1"].get(c,0) for c in CLASS_NAMES]
    vals_b  = [m_b["overall_accuracy"], m_b["macro_f1"]] + [m_b["per_class_f1"].get(c,0) for c in CLASS_NAMES]
    x, w    = np.arange(len(names)), 0.38
    fig, ax = plt.subplots(figsize=figsize)
    b_a = ax.bar(x-w/2, vals_a, w, color=TEAL,  label=label_a, zorder=3)
    b_b = ax.bar(x+w/2, vals_b, w, color=AMBER, label=label_b, zorder=3)
    for bars in (b_a, b_b):
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7, color=NAVY)
    ax.axvline(1.5, color=GREY, ls="--", lw=1, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8)
    ax.set_ylim(0, 1.15); ax.set_ylabel("Score", color=NAVY)
    ax.set_title(f"Model Performance: {label_a} vs {label_b}",
                 fontsize=14, fontweight="bold", color=NAVY, pad=12)
    ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.4, zorder=0)
    plt.tight_layout(); plt.show()

def plot_confidence_distribution(probs_a, probs_b, label_a="HAM10000", label_b="African Context", figsize=(8,4)):
    _style()
    ca = [max(p) for p in probs_a]; cb = [max(p) for p in probs_b]
    fig, ax = plt.subplots(figsize=figsize)
    ax.hist(ca, bins=30, color=TEAL,  alpha=0.65, label=label_a, density=True)
    ax.hist(cb, bins=30, color=AMBER, alpha=0.65, label=label_b, density=True)
    ax.axvline(0.8, color=RED, ls="--", lw=1.2, label="Threshold 0.80")
    ax.set_xlabel("Max Confidence Score", color=NAVY)
    ax.set_ylabel("Density", color=NAVY)
    ax.set_title("Confidence Distribution — Both Datasets",
                 fontsize=13, fontweight="bold", color=NAVY, pad=10)
    ax.legend(fontsize=9)
    ax.text(0.72, ax.get_ylim()[1]*0.85,
            "⚠ High confidence
does not mean
correct",
            fontsize=8, color=AMBER, ha="center",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=AMBER, alpha=0.8))
    plt.tight_layout(); plt.show()

def display_failure_grid(images, pred_names, true_names, confidences,
                          fitzpatrick_types=None, n=8, title="Failures", figsize=(14,7)):
    _style()
    n = min(n, len(images))
    cols, rows = 4, (n+3)//4
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    MEAN = np.array([0.485,0.456,0.406]); STD = np.array([0.229,0.224,0.225])
    for i in range(n):
        ax  = axes[i]; img = images[i]
        if isinstance(img, torch.Tensor):
            img = img.numpy()
        if isinstance(img, np.ndarray) and img.ndim == 3 and img.shape[0] == 3:
            img = np.transpose(img, (1,2,0))
            img = (img * STD + MEAN).clip(0,1)
        ax.imshow(img); ax.axis("off")
        colour = RED if pred_names[i] != true_names[i] else TEAL
        fitz   = f"
Fitzpatrick {fitzpatrick_types[i]}" if fitzpatrick_types and fitzpatrick_types[i] else ""
        ax.set_title(f"True: {true_names[i]}", fontsize=7, color=TEAL, pad=2)
        ax.text(0.5, -0.05, f"Pred: {pred_names[i]} ({confidences[i]:.0%}){fitz}",
                transform=ax.transAxes, fontsize=7, color=colour, ha="center", va="top")
    for j in range(n, len(axes)): axes[j].axis("off")
    fig.suptitle(title, fontsize=13, fontweight="bold", color=NAVY, y=1.01)
    plt.tight_layout(); plt.show()

def plot_options_comparison(figsize=(8,4)):
    _style()
    options    = ["Option A
Collect & Fine-tune", "Option B
Restricted + Human Review",
                  "Option C
Halt Deployment"]
    dimensions = ["Time to
Deployment", "Data Collection
Burden",
                  "Risk to
Patient", "Risk to
Startup"]
    values = np.array([[2,2,1,1],[1,0,1,1],[0,0,0,2]])
    labels = np.array([["High","High","Med","Low"],
                       ["Med", "Low", "Med","Med"],
                       ["—",   "—",   "Low","High"]])
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(values, cmap=plt.cm.RdYlGn_r, norm=plt.Normalize(0,2), aspect="auto")
    ax.set_xticks(range(4)); ax.set_xticklabels(dimensions, fontsize=9, color=NAVY)
    ax.set_yticks(range(3)); ax.set_yticklabels(options, fontsize=9, color=NAVY)
    ax.xaxis.set_ticks_position("top"); ax.xaxis.set_label_position("top")
    for i in range(3):
        for j in range(4):
            ax.text(j, i, labels[i,j], ha="center", va="center", fontsize=11,
                    fontweight="bold", color="white" if values[i,j]==2 else NAVY)
    ax.set_title("Deployment Options — Risk and Burden Comparison",
                 fontsize=12, fontweight="bold", color=NAVY, pad=30)
    handles = [mpatches.Patch(color=c, label=l)
               for c, l in [("#4CAF50","Low"),("#FFC107","Med"),("#F44336","High")]]
    ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(1.0,-0.18),
              ncol=3, fontsize=8, frameon=True)
    plt.tight_layout(); plt.show()

def plot_fitzpatrick_distribution(distribution, dataset_name="Dataset", figsize=(7,4)):
    _style()
    types = sorted(distribution)
    total = sum(distribution.values())
    pcts  = [distribution[t]/total*100 for t in types]
    cols  = [AMBER if t >= 5 else TEAL for t in types]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar([f"Type {t}" for t in types], pcts, color=cols, edgecolor="white")
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{pct:.1f}%", ha="center", va="bottom", fontsize=9, color=NAVY)
    ax.set_ylabel("% of Dataset", color=NAVY)
    ax.set_title(f"Fitzpatrick Skin Type Distribution — {dataset_name}",
                 fontsize=12, fontweight="bold", color=NAVY, pad=10)
    handles = [mpatches.Patch(color=TEAL, label="Types I–IV"),
               mpatches.Patch(color=AMBER, label="Types V–VI (underrepresented)")]
    ax.legend(handles=handles, fontsize=8)
    ax.set_ylim(0, max(pcts)*1.3)
    plt.tight_layout(); plt.show()

print(f"✓ All helpers loaded | device = ", end="")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(DEVICE)

In [ ]:
# Configuration — update paths if the facilitator uses a different folder layout
BATCH_SIZE        = 32
CONFIDENCE_THRESH = 0.80
NUM_WORKERS       = 2   # set to 0 if you get DataLoader errors on Windows/Mac

HAM_IMAGES  = DATA_DIR / "ham10000_test"  / "images"
HAM_CSV     = DATA_DIR / "ham10000_test"  / "HAM10000_metadata.csv"
AFR_IMAGES  = DATA_DIR / "african_context" / "images"
AFR_CSV     = DATA_DIR / "african_context" / "african_context_labels.csv"
MODEL_PATH  = DATA_DIR / "model"          / "efficientnet_b0_ham10000.pth"
DATASHEET   = Path("IndabaX-2026/Ethical Computer Vision/data/ham10000_datasheet.md")

# Check what's available
for name, path in [("HAM10000 images", HAM_IMAGES), ("HAM10000 CSV", HAM_CSV),
                   ("African images",  AFR_IMAGES), ("African CSV",  AFR_CSV),
                   ("Model checkpoint", MODEL_PATH)]:
    status = "✓" if Path(path).exists() else "✗ NOT FOUND"
    print(f"  {status}  {name}: {path}")

---
## 📊 SECTION 01 — THE BASELINE RUN
*The model has strong benchmark numbers. The startup's CTO cited 87% accuracy. Let's verify.*

---

Everything is loaded. The model was trained on 10,015 dermatoscopic images collected primarily from clinics in Austria and Australia. It has never seen a phone-camera image. It has never been evaluated on dark skin. The benchmark number is real — but what does it measure, and on whom?


In [ ]:
# Load the model
if MODEL_PATH.exists():
    model = load_model(str(MODEL_PATH), device=DEVICE)
else:
    print("⚠ Model checkpoint not found — update MODEL_PATH or download the data.")
    model = None

In [ ]:
# Sanity check — forward pass on a noise tensor
if model:
    noise = torch.randn(1, 3, 224, 224).to(DEVICE)
    with torch.no_grad():
        logits = model(noise)
        probs  = torch.softmax(logits, 1).squeeze().cpu().numpy()
    top_idx = probs.argmax()
    print(f"Sanity check passed — output shape: {logits.shape}")
    print(f"Top prediction on noise: {CLASS_LABELS_FLAT[CLASS_NAMES[top_idx]]} ({probs[top_idx]:.1%})")
    del noise

*⚠ The next cell loads HAM10000. Expect a few seconds on first load.*

In [ ]:
# Load HAM10000 test split
if HAM_IMAGES.exists() and HAM_CSV.exists():
    ham_loader, ham_ds = make_loader(HAM_IMAGES, HAM_CSV, BATCH_SIZE, NUM_WORKERS)
    print(f"✓ HAM10000 test split: {len(ham_ds)} images")
    # Class distribution
    from collections import Counter
    dist = Counter(CLASS_NAMES[l] for l in ham_ds.labels)
    print("  Class distribution:")
    for cls in CLASS_NAMES:
        print(f"    {cls:>6s} ({CLASS_LABELS_FLAT[cls]:<25s}): {dist.get(cls,0)}")
else:
    ham_loader = ham_ds = None
    print("⚠ HAM10000 data not found — skipping.")

*⚠ Running inference on ~2,000 images. CPU: 3–5 min. T4 GPU: under 1 min.*

In [ ]:
# Run inference on HAM10000 test split
if model and ham_loader:
    ham_results  = run_inference(model, ham_loader, DEVICE)
    ham_metrics  = compute_metrics(
        ham_results["predictions"],
        ham_results["true_labels"],
        ham_results["probabilities"]
    )
    print(f"\nHAM10000 results:")
    print(f"  Overall accuracy : {ham_metrics['overall_accuracy']:.1%}")
    print(f"  Macro F1         : {ham_metrics['macro_f1']:.3f}")
    print(f"  Weighted F1      : {ham_metrics['weighted_f1']:.3f}")
    print(f"  Avg confidence when correct : {ham_metrics['avg_confidence_correct']:.1%}")
    print(f"  Avg confidence when wrong   : {ham_metrics['avg_confidence_wrong']:.1%}")
else:
    ham_results = ham_metrics = None
    print("⚠ Skipped — no model or data.")

In [ ]:
# Normalised confusion matrix
if ham_metrics:
    plot_confusion_matrix(
        ham_results["true_labels"],
        ham_results["predictions"],
        title="Confusion Matrix — HAM10000 Test Split"
    )

In [ ]:
# Per-class F1
if ham_metrics:
    plot_per_class_f1(ham_metrics, title="Per-Class F1 — HAM10000 Test Split")

In [ ]:
# Display the 5 most confident CORRECT predictions — the model at its best
if ham_results and ham_ds:
    table = build_failure_table(
        ham_results["predictions"], ham_results["true_labels"],
        ham_results["probabilities"], ham_results["image_ids"],
        conf_thresh=CONFIDENCE_THRESH
    )
    correct = table[table["error_type"] == "correct"].head(5)

    print("Top 5 most confident CORRECT predictions (the model at its best):")
    print(correct[["image_id","true_label","predicted_label","confidence"]].to_string(index=False))

In [ ]:
# Display the 5 most confident WRONG predictions — the model at its worst
if ham_results and ham_ds:
    wrong = table[table["error_type"] != "correct"].head(5)
    print("Top 5 most confident WRONG predictions (high-stakes failures):")
    print(wrong[["image_id","true_label","predicted_label","confidence","error_type"]].to_string(index=False))
    print()
    print(f"Total high-confidence wrong predictions (conf ≥ {CONFIDENCE_THRESH:.0%}): "
          f"{(table['error_type']=='high_conf_wrong').sum()}")

---
#### ✍️ Reflection 1 — First Impressions

*Write your answers in the cell below. There is no right answer — this is your analysis.*

1. The model reports a headline accuracy figure. Is that number trustworthy as a safety metric? Why or why not?
2. Which class has the lowest F1 score? What does that mean clinically if this model is used for screening?
3. The model is highly confident on its worst predictions. Why is this dangerous in a medical context?

---

*Your answer here.*

---
## 🔬 SECTION 02 — THE STRESS TEST
*Same model. Same code. Different images.*

---

The next dataset was collected from phones — Android devices, outdoor conditions, varying lighting — from patients in West African clinical settings. The images are phone-camera photographs, not dermatoscope images. The conditions are the same as the community health workers will encounter.

The model has never seen images like these.


In [ ]:
# Load the African-context dataset
if AFR_IMAGES.exists() and AFR_CSV.exists():
    afr_loader, afr_ds = make_loader(AFR_IMAGES, AFR_CSV, BATCH_SIZE, NUM_WORKERS)
    print(f"✓ African-context dataset: {len(afr_ds)} images")
    from collections import Counter
    dist_afr = Counter(CLASS_NAMES[l] for l in afr_ds.labels)
    print("  Class distribution:")
    for cls in CLASS_NAMES:
        print(f"    {cls:>6s} ({CLASS_LABELS_FLAT[cls]:<25s}): {dist_afr.get(cls,0)}")
    # Fitzpatrick distribution if available
    fitz_vals = [f for f in afr_ds.fitzpatrick if f is not None]
    if fitz_vals:
        from collections import Counter as C
        print(f"  Fitzpatrick types: {dict(sorted(C(fitz_vals).items()))}")
else:
    afr_loader = afr_ds = None
    print("⚠ African-context data not found — skipping.")

In [ ]:
# Run inference on African-context dataset
if model and afr_loader:
    afr_results = run_inference(model, afr_loader, DEVICE)
    afr_metrics = compute_metrics(
        afr_results["predictions"],
        afr_results["true_labels"],
        afr_results["probabilities"]
    )
    print("\nAfrican-context results:")
    print(f"  Overall accuracy : {afr_metrics['overall_accuracy']:.1%}")
    print(f"  Macro F1         : {afr_metrics['macro_f1']:.3f}")
    print(f"  Weighted F1      : {afr_metrics['weighted_f1']:.3f}")
    if ham_metrics:
        acc_gap = ham_metrics['overall_accuracy'] - afr_metrics['overall_accuracy']
        f1_gap  = ham_metrics['macro_f1'] - afr_metrics['macro_f1']
        print(f"\n  Performance gap vs HAM10000:")
        print(f"    Accuracy gap : {acc_gap:+.1%}")
        print(f"    Macro F1 gap : {f1_gap:+.3f}")
else:
    afr_results = afr_metrics = None
    print("⚠ Skipped — no model or data.")

In [ ]:
# Side-by-side comparison — the central visual of the audit
if ham_metrics and afr_metrics:
    plot_metric_comparison(
        ham_metrics, afr_metrics,
        label_a="HAM10000 (dermatoscope)",
        label_b="African Context (phone camera)"
    )

In [ ]:
# Confidence distribution — does the model know when it is wrong?
if ham_results and afr_results:
    plot_confidence_distribution(
        ham_results["probabilities"],
        afr_results["probabilities"],
        label_a="HAM10000",
        label_b="African Context"
    )

In [ ]:
# Display the 8 worst failures from the African-context dataset
if afr_results and afr_ds:
    afr_table = build_failure_table(
        afr_results["predictions"], afr_results["true_labels"],
        afr_results["probabilities"], afr_results["image_ids"],
        conf_thresh=CONFIDENCE_THRESH
    )
    worst_ids = afr_table[afr_table["error_type"] != "correct"].head(8)["image_id"].tolist()

    # Load the actual images for display
    id_to_idx = {img_id: i for i, img_id in enumerate(afr_ds.image_ids)}
    show_imgs, show_preds, show_trues, show_confs, show_fitz = [], [], [], [], []
    for img_id in worst_ids:
        if img_id in id_to_idx:
            idx = id_to_idx[img_id]
            tensor, label, _ = afr_ds[idx]
            row = afr_table[afr_table["image_id"]==img_id].iloc[0]
            show_imgs.append(tensor)
            show_preds.append(row["predicted_label"])
            show_trues.append(row["true_label"])
            show_confs.append(row["confidence"])
            show_fitz.append(afr_ds.fitzpatrick[idx])

    if show_imgs:
        display_failure_grid(show_imgs, show_preds, show_trues, show_confs,
                              fitzpatrick_types=show_fitz, n=len(show_imgs),
                              title="Worst Failures — African-Context Dataset")

---
#### ✍️ Reflection 2 — The Performance Gap

Fill in from the metrics you observed:

| Metric | HAM10000 | African Context | Gap |
|--------|----------|-----------------|-----|
| Overall Accuracy | | | |
| Macro F1 | | | |
| Melanoma F1 | | | |

1. The model is similarly confident on both datasets. Accuracy drops sharply on the African-context set. What does this tell you about the model's calibration?
2. Which error type is clinically most dangerous — false negatives (missing melanoma) or false positives (flagging a benign lesion)? Does your answer change in the community health worker context?
3. A colleague argues that the performance gap is just because phone photos are lower quality — not a bias issue. How would you respond?

---

*Your answer here.*

---
## 🔍 SECTION 03 — DOCUMENT FAILURES & ROOT CAUSE
*The failure is real. Now we trace it. Where did it come from?*

---

We have seen that the model performs worse on African-context images. Two questions remain: which failures are clinically most dangerous, and why is the model failing? The answers are in the training data — not in the model architecture.


**Part A — Document the Failures**

In [ ]:
# Structured failure table — one row per image
if afr_results:
    print(f"African-context dataset — failure breakdown:")
    print()
    summary = afr_table.groupby("error_type").size().reset_index(name="count")
    summary["% of total"] = (summary["count"] / len(afr_table) * 100).round(1)
    print(summary.to_string(index=False))

In [ ]:
# YOUR CODE — filter to high-confidence wrong predictions
# These are the most dangerous: the model is certain, and wrong.
if afr_results:
    high_conf_wrong = afr_table[
        (afr_table["error_type"] == "high_conf_wrong") &
        (afr_table["error_type"] != "correct")
    ].head(10)
    print(f"High-confidence wrong predictions (conf ≥ {CONFIDENCE_THRESH:.0%}):")
    print(high_conf_wrong[["image_id","true_label","predicted_label","confidence"]].to_string(index=False))
    print(f"\nTotal: {len(high_conf_wrong)} shown | "
          f"{(afr_table['error_type']=='high_conf_wrong').sum()} in full dataset")

In [ ]:
# Error type summary with clinical interpretation
if afr_results:
    errors_only = afr_table[afr_table["error_type"] != "correct"]
    print("Error type summary (errors only):")
    print()
    type_counts = errors_only["error_type"].value_counts()
    for etype, count in type_counts.items():
        pct = count / len(errors_only) * 100
        clinical = {
            "false_negative":  "⚠ MISSED MALIGNANCY — highest clinical risk",
            "false_positive":  "unnecessary referral / patient anxiety",
            "high_conf_wrong": "⚠ CONFIDENT WRONG — model will not flag for review",
            "wrong_other":     "misclassification between benign classes",
        }.get(etype, "")
        print(f"  {etype:<20s}  {count:>4d}  ({pct:.1f}%)  {clinical}")

**Part B — Root Cause**

The failure is documented. Now we trace it. The model did not fail because of a bug or a misconfigured hyperparameter. It failed because of what was — and was not — in the training data.

Read the HAM10000 datasheet below. Pay attention to the collection geography, the imaging equipment, and the demographic notes.


In [ ]:
# Display the HAM10000 datasheet
if DATASHEET.exists():
    from IPython.display import Markdown, display
    display(Markdown(DATASHEET.read_text()))
else:
    print("⚠ Datasheet not found at:", DATASHEET)
    print("It should be at IndabaX-2026/Ethical Computer Vision/data/ham10000_datasheet.md")
    print()
    print("Summary of key facts about HAM10000:")
    print("  - Collected in Austria (ViDIR group, Medical University of Vienna) and Australia (QIMR Berghofer)")
    print("  - Images acquired with dermatoscopes — not phone cameras")
    print("  - Estimated Fitzpatrick distribution: ~70% Type I–II, ~25% Type III, ~5% Type IV–VI")
    print("  - No systematic demographic metadata collected per patient")
    print("  - No African sites included in collection")

In [ ]:
# Fitzpatrick distribution — estimated for HAM10000 vs measured for African-context
ham_fitz_estimated = {1: 35, 2: 35, 3: 20, 4: 7, 5: 2, 6: 1}  # estimated from collection geography

plot_fitzpatrick_distribution(ham_fitz_estimated, dataset_name="HAM10000 (estimated)")

if afr_ds:
    fitz_vals = [f for f in afr_ds.fitzpatrick if f is not None]
    if fitz_vals:
        from collections import Counter
        afr_fitz = dict(Counter(fitz_vals))
        plot_fitzpatrick_distribution(afr_fitz, dataset_name="African Context (measured)")

In [ ]:
# Side-by-side: HAM10000 dermatoscope image vs African-context phone image
# Visual demonstration of the acquisition quality gap
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
sample_images = [
    (HAM_IMAGES,  "HAM10000 — Dermatoscope Image
(standardised, high resolution, controlled lighting)"),
    (AFR_IMAGES,  "African-Context — Phone Camera Image
(outdoor, variable lighting, lower resolution)"),
]
for ax, (img_dir, caption) in zip(axes, sample_images):
    img_dir = Path(img_dir)
    if img_dir.exists():
        sample = next(img_dir.glob("*.jpg"), None)
        if sample:
            ax.imshow(plt.imread(str(sample)))
            ax.set_title(caption, fontsize=10, color=NAVY)
            ax.axis("off")
        else:
            ax.text(0.5, 0.5, "No images found", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
    else:
        ax.text(0.5, 0.5, "Directory not found", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
plt.suptitle("Acquisition Gap: Training Distribution vs Deployment Reality",
             fontsize=13, fontweight="bold", color=NAVY)
plt.tight_layout()
plt.show()

---
#### ✍️ Reflection 3 — Tracing the Failure

You have now read the HAM10000 datasheet.

1. The datasheet lists the collection sites. Which countries are represented? Which continent is absent?
2. The model was fine-tuned on HAM10000. Explain — in plain language, without jargon — why a model trained on this data would perform worse on images from West Africa.
3. Is this a problem with the model architecture? With the training code? Or with something more fundamental? What would you change?
4. A lawyer argues the startup is liable if the model misses a melanoma. A technologist argues the model is working as designed — it just wasn't designed for this use case. Who is right?

---

*Your answer here.*

---
## 🛤️ SECTION 04 — PROPOSE A PATH
*Analysis is not enough. An auditor's job is to make a recommendation.*

---

There is no single right answer. Three credible paths exist, each with real trade-offs. Your job is to pick one and defend it.


> ⚖️ **Three paths forward:**

**Option A — Collect and fine-tune**
Gather 500–1,000 images of skin conditions from African clinical settings with proper consent and Fitzpatrick labels. Fine-tune the model on this data. Re-audit before deployment. Estimated timeline: 6–12 months.

*Trade-off:* takes time and resources; the right thing to do for long-term safety. Requires community partnerships and ethical data collection protocols.

**Option B — Restricted deployment with mandatory human review**
Deploy immediately but restrict the model to a decision-support role only. Every prediction above the melanoma threshold triggers a mandatory dermatologist review before any clinical action is taken. The model cannot act autonomously.

*Trade-off:* allows faster deployment; reduces but does not eliminate harm; adds operational cost; still exposes patients to a biased tool.

**Option C — Halt and rebuild**
Do not deploy. Return to the startup with a documented audit report. Recommend starting from a more representative dataset (Fitzpatrick17k + SCIN) and rebuilding the model before returning to deployment discussion.

*Trade-off:* the safest option for patients; significant commercial cost to the startup; delays access to any tool for community health workers.


In [ ]:
# Cost-benefit comparison across three options
plot_options_comparison()

---

**Real-world precedents:**

**Option A — The SCIN template.** In 2024, Google Research released the Skin Condition Image Network (SCIN) — 10,408 diverse skin images collected with explicit Fitzpatrick labels from a global community of contributors. This is what responsible collection at scale looks like. [Read more →](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/)

**Option B — The FDA framework.** The US FDA's guidance on AI/ML-based Software as a Medical Device (SaMD) requires pre-specified performance goals on demographically representative test sets. A human-in-the-loop requirement is one path to approval for tools that do not yet meet those goals.

**Option C — The pulse oximeter precedent.** Pulse oximeters were deployed for decades before Sjoding et al. (NEJM 2020) showed they systematically over-read on darker skin — leading to COVID-19 patients being denied oxygen they needed. The cost of that failure vastly exceeded the cost of earlier detection. [Read the paper →](https://www.nejm.org/doi/full/10.1056/NEJMc2029240)

---

#### ✍️ Reflection 4 — Your Recommendation

Select your recommended path and defend it in 3–5 sentences.

**Option A / B / C** *(delete two)*

**Recommendation:**
*Write here.*

**Justification:**
*Write here — reference the specific metrics you observed.*

**Conditions:**
*What conditions must be met before any deployment? What metrics would you require the startup to achieve?*

---

*Your answer here.*

---

## Conclusion

The startup's CTO is waiting for your answer. The health workers in Brong-Ahafo are waiting for the app. These are not abstract trade-offs — they are decisions that will affect real patients.

What you have done in this notebook is a real audit. The gap you measured is real. The data bias you traced is real. The three options are real options that medical AI teams face.

The technical skills you used — loading a model, running inference, computing metrics, building failure tables, tracing bias to training data — are the same skills used in actual ML safety work. The difference between a responsible deployment and a harmful one is often just whether someone ran these cells and acted on what they found.

---

📚 **Key references from this audit:**

| Resource | Why |
|---|---|
| [HAM10000 — Harvard Dataverse](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T) | The training dataset — read the metadata |
| [Fitzpatrick17k](https://github.com/mattgroh/fitzpatrick17k) | The diversity benchmark — what HAM10000 should have been |
| [SCIN Dataset — Google 2024](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/) | How to collect representative skin data responsibly |
| [Racial Bias in Pulse Oximetry — NEJM 2020](https://www.nejm.org/doi/full/10.1056/NEJMc2029240) | The cost of deploying a biased medical device |
| [Model Cards — Mitchell et al. 2018](https://arxiv.org/abs/1810.03993) | What documentation should accompany every model |
| [Datasheets for Datasets — Gebru et al. 2018](https://arxiv.org/abs/1803.09010) | What documentation should accompany every dataset |
| [FDA AI/ML SaMD Guidance](https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-and-machine-learning-software-medical-device) | The regulatory framework for medical AI |

---

*Part 1 of this tutorial — led by Nana Sam Yeboah — covers the computer vision foundations behind this model: how images become tensors, how preprocessing pipelines work, how CNNs learn, and how YOLO, Vision Transformers, and CLIP extend the toolkit.*
